# Smart Irrigation IoT - EDA, Pembersihan Data, Pemodelan, dan Evaluasi

Kasus 07: Irigasi Pintar Berbasis IoT (16 Petak Rumah Kaca, Lembang).
Metrik utama: F1-Score kelas positif (`perlu_siram = 1`).
Seluruh fungsi pembersihan data dan rekayasa fitur menggunakan modul `src/features.py` agar konsisten dengan API dan aplikasi Streamlit.


In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix, classification_report

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import FEATURE_COLUMNS, clean_raw_data, build_batch_features
from src.train import train_and_save_artifacts, MODEL_PATH, METADATA_PATH

sns.set_theme(style="whitegrid")
print("Project root:", PROJECT_ROOT)


## 1. Memuat dan Memeriksa Data Mentah


In [ ]:
raw_path = PROJECT_ROOT / "data" / "raw.csv.gz"
raw_df = pd.read_csv(raw_path)
print(f"Ukuran data mentah: {raw_df.shape[0]:,} baris x {raw_df.shape[1]} kolom")
raw_df.head()


In [ ]:
# Pembersihan data melalui src/features.py
df_clean, cleaning_stats = clean_raw_data(raw_df)
pd.DataFrame([
    {"Masalah": k, "Jumlah Baris": v} for k, v in cleaning_stats.items()
])


## 2. Rekayasa Fitur dan Pembagian Data Berbasis Waktu

Pembagian data menggunakan *time-based split* agar model tidak melihat informasi masa depan:
- Train: 1 Agustus - 30 September 2025
- Validation: 1 - 14 Oktober 2025
- Test: 15 - 31 Oktober 2025 (hanya dibuka satu kali pada evaluasi akhir)


In [ ]:
df = build_batch_features(df_clean)
df = df.sort_values(["waktu_dt", "petak"]).reset_index(drop=True)

mask_train = (df["waktu_dt"] >= "2025-08-01 00:00:00") & (df["waktu_dt"] <= "2025-09-30 23:59:59")
mask_val = (df["waktu_dt"] >= "2025-10-01 00:00:00") & (df["waktu_dt"] <= "2025-10-14 23:59:59")
mask_test = (df["waktu_dt"] >= "2025-10-15 00:00:00") & (df["waktu_dt"] <= "2025-10-31 23:59:59")

train_df = df.loc[mask_train].reset_index(drop=True)
val_df = df.loc[mask_val].reset_index(drop=True)
test_df = df.loc[mask_test].reset_index(drop=True)

print(f"Train: {len(train_df):,} | Validation: {len(val_df):,} | Test: {len(test_df):,}")
assert train_df["waktu_dt"].max() < val_df["waktu_dt"].min() < test_df["waktu_dt"].min()


## 3. EDA Singkat pada Data Train


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

siram_crop = train_df.groupby(["tanaman", "jenis_tanah"])["perlu_siram"].mean().reset_index()
siram_crop["persen"] = siram_crop["perlu_siram"] * 100
sns.barplot(data=siram_crop, x="tanaman", y="persen", hue="jenis_tanah", ax=axes[0])
axes[0].set_title("Proporsi Perlu Siram (%) per Tanaman dan Jenis Tanah")
axes[0].set_ylabel("Persentase (%)")

train_siram = train_df[(train_df["perlu_siram"] == 1) & (train_df["kt_diff1"] <= 10.0)]
sns.boxplot(data=train_siram, x="tanaman", y="kelembaban_tanah_pct", hue="jenis_tanah", ax=axes[1])
axes[1].axhline(35.0, color="red", linestyle="--", label="Baseline 35%")
axes[1].set_title("Kelembaban Tanah Saat Perlu Siram (Train)")
axes[1].legend()

plt.tight_layout()
plt.show()


## 4. Baseline dan Pemilihan Model pada Validation Set


In [ ]:
X_train, y_train = train_df[FEATURE_COLUMNS].astype(np.float32), train_df["perlu_siram"].astype(int)
X_val, y_val = val_df[FEATURE_COLUMNS].astype(np.float32), val_df["perlu_siram"].astype(int)

pred_val_base = (val_df["kelembaban_tanah_pct"] < 35.0).astype(int)

dt = DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=2026).fit(X_train, y_train)
rf = RandomForestClassifier(
    n_estimators=180, max_depth=12, min_samples_leaf=3,
    class_weight="balanced", random_state=2026, n_jobs=-1
).fit(X_train, y_train)
hgb = HistGradientBoostingClassifier(
    max_depth=6, learning_rate=0.06, l2_regularization=1.0,
    max_iter=250, min_samples_leaf=15, random_state=2026
).fit(X_train, y_train)

prob_val_rf = rf.predict_proba(X_val)[:, 1]
prob_val_hgb = hgb.predict_proba(X_val)[:, 1]
prob_val_ens = 0.5 * prob_val_rf + 0.5 * prob_val_hgb

val_summary = pd.DataFrame([
    {"Model": "Baseline (< 35%)", "F1 Validation": f1_score(y_val, pred_val_base)},
    {"Model": "Decision Tree", "F1 Validation": f1_score(y_val, dt.predict(X_val))},
    {"Model": "Random Forest (th=0.61)", "F1 Validation": f1_score(y_val, (prob_val_rf >= 0.61).astype(int))},
    {"Model": "HistGradientBoosting (th=0.47)", "F1 Validation": f1_score(y_val, (prob_val_hgb >= 0.47).astype(int))},
    {"Model": "Ensemble RF + HistGB (th=0.54)", "F1 Validation": f1_score(y_val, (prob_val_ens >= 0.54).astype(int))},
])
val_summary.round(4)


## 5. Pelatihan Final, Evaluasi Test Set, dan Penyimpanan Artifact

Fungsi `train_and_save_artifacts()` dari `src/train.py` melatih ulang model final pada gabungan *Train + Validation*, mengevaluasinya satu kali pada *Test Set*, dan menyimpan `artifacts/model.joblib` serta `artifacts/metadata.json`.


In [ ]:
bundel = train_and_save_artifacts()
mt = bundel["metrics_test"]

test_summary = pd.DataFrame([
    {"Model": "Baseline (< 35%)", "F1 Test": mt["f1_baseline"], "Precision": mt["prec_baseline"], "Recall": mt["rec_baseline"]},
    {"Model": "Random Forest (th=0.61)", "F1 Test": mt["f1_ml_rf"], "Precision": mt["prec_ml_rf"], "Recall": mt["rec_ml_rf"]},
    {"Model": "HistGradientBoosting (th=0.47)", "F1 Test": mt["f1_ml_hgb"], "Precision": np.nan, "Recall": np.nan},
    {"Model": "Ensemble RF + HistGB (th=0.54)", "F1 Test": mt["f1_ml_ensemble"], "Precision": mt["prec_ml_ensemble"], "Recall": mt["rec_ml_ensemble"]},
])
print("Artifact tersimpan di:", MODEL_PATH)
print("Metadata tersimpan di:", METADATA_PATH)
test_summary.round(4)


In [ ]:
# Tampilkan ringkasan metadata yang tersimpan
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
print(json.dumps(metadata, indent=2, ensure_ascii=False))
